In [ ]:
from PIL import Image
import numpy as np

In [ ]:
img = Image.open('img/2026-06-06-01.jpeg')

In [ ]:
img

#### 1. convert image to grayscale

In [ ]:
img_gray = img.convert('LA')
plt.imshow(img_gray)

#### 2. convert data into numpy matrix

In [ ]:
img_matrix = np.array(list(img_gray.getdata(band=0)), dtype=float)
# x = cols = img_gray.size[1], y = rows = img_gray.size[0]
img_matrix.shape = (img_gray.size[1], img_gray.size[0])
# img_matrix = np.matrix(img_matrix)
_ = plt.imshow(img_matrix, cmap='gray')

In [ ]:
img_gray.size

In [ ]:
img_matrix.shape

#### 3. calculate SVD of the image

In [ ]:
U, sigma, V = np.linalg.svd(img_matrix)

In [ ]:
U.shape, sigma.shape, V.shape

In [ ]:
rng = list(2**i for i in range(1, 8))
rng

In [ ]:
for i in rng:
    reconstructed_img = np.array(U[:, :i]) @ np.diag(sigma[:i]) @ np.array(V[:i, :])
    plt.imshow(reconstructed_img, cmap='gray')
    title = "n = %s" % i
    plt.title(title)
    plt.show()

#### 4. check the size of orginal image and compressed image

In [ ]:
# orginal image
img_matrix.shape, img_matrix.size

In [ ]:
# compressed image
m = 64 # number of singular values to keep
reconstructed_img = np.array(U[:, :m]) @ np.diag(sigma[:m]) @ np.array(V[:m, :])
plt.imshow(reconstructed_img, cmap='gray')
plt.show()

Instead of storing the full 183 x 275 matrix, we store only the first n = 64 components of the SVD factorization:

U matrix: Keeps the first 64 columns. Original is 183 x 275, truncated is 183 x 64. → Stores 183 * 64 = 11,712 numbers.

Σ (Sigma) matrix: Keeps the first 64 singular values. Since it's a diagonal matrix, only the 64 values on the diagonal are stored, not the full 183 x 275 grid. → Stores 64 numbers.

V^T matrix: Keeps the first 64 rows. Original is 183 x 275, truncated is 64 x 275. → Stores 64 * 275 = 17,600 numbers.

Total compressed size = 11,712 (U) + 64 (Σ) + 17,600 (V^T) = 29,376 numbers.

In [ ]:
np.array(U[:, :m]).size, np.diag(sigma[:m]).shape[0], np.array(V[:m, :]).size

In [ ]:
np.array(U[:, :m]).shape, np.diag(sigma[:m]).shape, np.array(V[:m, :]).shape

In [ ]:
183 * 64 + 64 + 64 * 275

In [ ]:
compressed_size = np.array(U[:, :m]).size + np.diag(sigma[:m]).shape[0] + np.array(V[:m, :]).size
compressed_size

In [ ]:
print(f'{compressed_size * 100 / img_matrix.size:.2f}%')
print(f'{img_matrix.size / compressed_size:.2f}x smaller')